# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)
- DOI: [10.71728/senscience.vcs2-05nj](https://sen.science/doi/10.71728/senscience.vcs2-05nj)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
# Access metadata (not subscripting as per instructions)
meta = dataset.metadata
print(f"Dataset Title: {meta.name}")
print(f"Description: {meta.description}")

## 2. Data Overview
Review available record sets in the dataset and preview a small subset of records.

All entity references below use their Croissant `@id`. Use these IDs in later steps for data extraction.


In [ ]:
# List all available record sets using their @id
record_sets = dataset.list_record_sets()
print("Available Record Sets (by @id):")
for rs in record_sets:
    print(f"  - {rs['@id']}: {rs['name']}")

# For demonstration, select the main survey record set
# (The actual @id may vary; fetch the one with description mentioning main survey responses)
main_rs = None
for rs in record_sets:
    desc = rs.get('description','').lower()
    if 'survey' in desc or 'main' in desc or 'responses' in desc or 'participant' in desc:
        main_rs = rs['@id']
        break
if not main_rs:
    main_rs = record_sets[0]['@id']
print(f"\nSelected main record set for exploration: {main_rs}")

# List fields for the selected record set, by their @id
fields = dataset.list_fields(record_set=main_rs)
print(f"\nFields in record set {main_rs}:")
for field in fields:
    print(f"  - {field['@id']}: {field['name']} (type: {field['dataType']})")

# Preview the first 3 records (using the @id)
print(f"\nSample records from record set {main_rs}:")
for idx, rec in enumerate(dataset.records(record_set=main_rs)):
    print(json.dumps(rec, indent=2))
    if idx >= 2:
        break

## 3. Data Extraction
Load data from all record sets into Pandas DataFrames using their record set and field `@id`s. This enables structured tabular analysis and ensures reference consistency according to the Croissant schema.

In [ ]:
# Extract all available record sets into pandas DataFrames by their @id
dataframes = {}
for rs in record_sets:
    rs_id = rs['@id']
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records in record set: {rs_id}")
    else:
        print(f"No records found for record set: {rs_id}")

print("\nAvailable DataFrame keys (record set @id):")
print(list(dataframes.keys()))

# Example: show columns of the main record set
main_df = dataframes[main_rs]
print(f"\nColumns in DataFrame for record set {main_rs}:\n{main_df.columns.tolist()}")

# Show the first five rows
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering, normalization, and grouping, using field `@id`s for all references as specified by the Croissant schema. This section will demonstrate filtering records by a numeric field, normalization, and analyzing group differences.

In [ ]:
# Identify a numeric field @id for analysis, e.g., GAD-7 total scores
# We'll search for any field containing 'gad_7' or 'phq_9' in the @id or name.
numeric_field_id = None
for field in fields:
    fname = (field.get('@id','') + field.get('name','')).lower()
    if 'gad' in fname and 'score' in fname:
        numeric_field_id = field['@id']
        break
if not numeric_field_id:
    for field in fields:
        fname = (field.get('@id','') + field.get('name','')).lower()
        if 'phq' in fname and 'score' in fname:
            numeric_field_id = field['@id']
            break
# If still not found, pick first numeric field
if not numeric_field_id:
    for field in fields:
        if field.get('dataType', '').lower() in ['number', 'float', 'integer']:
            numeric_field_id = field['@id']
            break

if not numeric_field_id:
    raise Exception("No numeric field found for EDA.")
print(f"Selected numeric field for analysis: {numeric_field_id}")

# Filter records with numeric_field > a threshold
threshold = 10
if numeric_field_id in main_df.columns:
    filtered_df = main_df[pd.to_numeric(main_df[numeric_field_id], errors='coerce') > threshold].copy()
    print(f"Filtered records in {main_rs} where {numeric_field_id} > {threshold}: {len(filtered_df)} records")
    
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id].astype(float) - \
                                                     filtered_df[numeric_field_id].astype(float).mean()) / \
                                                    filtered_df[numeric_field_id].astype(float).std()
    print(f"First 5 normalized records:")
    display_cols = [numeric_field_id, f"{numeric_field_id}_normalized"]
    print(filtered_df[display_cols].head())
else:
    print(f"The numeric field {numeric_field_id} was not found in the DataFrame columns.")

# Group by a demographic field (e.g., 'level_of_education' or any field containing 'education' or 'gender') if present
group_field_id = None
for field in fields:
    n = field['@id'].lower() + field['name'].lower()
    if 'education' in n or 'gender' in n:
        group_field_id = field['@id']
        break
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} by group ({group_field_id}):")
    print(grouped_df)
else:
    print("No suitable group field found for aggregation.")

## 5. Visualization
Visualize the distribution of the selected numeric field, for example, the GAD-7/PHQ-9 total scores, and compare means across groups (e.g., by education or gender).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
if numeric_field_id in main_df.columns:
    plt.figure(figsize=(8,5))
    vals = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
    sns.histplot(vals.dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in {main_rs}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# If group field was found, plot group means
if 'grouped_df' in locals() and group_field_id is not None:
    plt.figure(figsize=(8,5))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=30)
    plt.show()


## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to load and explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, referencing all schema elements by their Croissant `@id`.

- We listed and loaded record sets by their IDs.
- Identified numeric fields (e.g., GAD-7, PHQ-9 scores) and performed exploratory analyses, filtering, and normalization using field `@id`.
- Produced basic distributional and group-level plots to aid interpretation.

This workflow serves as a template for reproducible, `@id`-based FAIR data analyses using Croissant-conformant datasets.
